In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
from PIL import Image, UnidentifiedImageError
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

# ✅ 1️⃣ Data Paths
data_dir = '/content/drive/MyDrive/cotton'  # Updae this path

# ✅ 2️⃣ Check & Remove Corrupt Images
def remove_corrupt_images(data_dir):
    corrupt_files = []
    for root, _, files in os.walk(data_dir):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                img = Image.open(file_path)
                img.verify()
            except (IOError, SyntaxError, UnidentifiedImageError):
                print(f'Corrupt image found: {file_path}')
                corrupt_files.append(file_path)

    for file_path in corrupt_files:
        os.remove(file_path)
        print(f'Removed {file_path}')

remove_corrupt_images(data_dir)

# ✅ 3️⃣ Data Transform & Loading
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder(root=data_dir, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# ✅ 4️⃣ Model Definition
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = resnet18(weights=ResNet18_Weights.DEFAULT)
num_classes = len(dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# ✅ 5️⃣ Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# ✅ 6️⃣ Training Loop
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Val Acc: {val_acc:.4f}')

# ✅ 7️⃣ Save Trained Model
torch.save(model.state_dict(), 'cotton_disease_classifier.pth')

# ✅ 8️⃣ Inference Function
def predict(image_path):
    image = Image.open(image_path)
    image = transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return dataset.classes[pred.item()]

# Example usage:
# disease = predict('/path/to/sample_leaf.jpg')
# print(f'Predicted disease: {disease}')


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 173MB/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [1/20], Loss: 1.7202, Val Acc: 0.5643
Epoch [2/20], Loss: 0.8383, Val Acc: 0.6301
Epoch [3/20], Loss: 0.5052, Val Acc: 0.6207
Epoch [4/20], Loss: 0.3189, Val Acc: 0.6332
Epoch [5/20], Loss: 0.2211, Val Acc: 0.6238
Epoch [6/20], Loss: 0.1953, Val Acc: 0.6364
Epoch [7/20], Loss: 0.1565, Val Acc: 0.6458
Epoch [8/20], Loss: 0.1208, Val Acc: 0.6364
Epoch [9/20], Loss: 0.1264, Val Acc: 0.6458
Epoch [10/20], Loss: 0.1164, Val Acc: 0.6395
Epoch [11/20], Loss: 0.1079, Val Acc: 0.6364
Epoch [12/20], Loss: 0.0868, Val Acc: 0.6426
Epoch [13/20], Loss: 0.0923, Val Acc: 0.6520
Epoch [14/20], Loss: 0.0926, Val Acc: 0.6552
Epoch [15/20], Loss: 0.0833, Val Acc: 0.6489
Epoch [16/20], Loss: 0.0835, Val Acc: 0.6364
Epoch [17/20], Loss: 0.0845, Val Acc: 0.6489
Epoch [18/20], Loss: 0.0742, Val Acc: 0.6458
Epoch [19/20], Loss: 0.0698, Val Acc: 0.6426
Epoch [20/20], Loss: 0.0731, Val Acc: 0.6395


In [ ]:
print(f"Number of classes: {len(dataset.classes)}")
print(f"Class names: {dataset.classes}")


Number of classes: 9
Class names: ['maize_aphid', 'maize_beetle', 'maize_fall_armyworm', 'maize_flea_beetles_disease', 'maize_foot_and_collar_rot', 'maize_mealybug', 'maize_stem_rot', 'maize_thrips_disease', 'maize_white_flies_disease']


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# ✅ After training loop finishes
y_true = []
y_pred = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# ✅ Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

# ✅ Classification Report (Precision, Recall, F1-score per class)
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=dataset.classes))


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Confusion Matrix:
[[ 7  0  0  0  2  2  5  3  0  3]
 [ 0 26  0  0  0  0  0  3  0  1]
 [ 0  0  3  0  0  1  1  1  0  5]
 [ 2  0  0 43  1  0  0  2  0  0]
 [ 0  2  0  1 23  4  5  0  4  2]
 [ 0  0  1  1  4 25  5  2  3  1]
 [ 4  0  0  0  1  2 22  0  3  1]
 [ 7  1  0  2  0  1  0 24  1  2]
 [ 1  0  0  0  0  4  1  0 28  1]
 [ 1  0  4  1  0  4  0  0  0  9]]

Classification Report:
                              precision    recall  f1-score   support

          cotton_anthracnose       0.32      0.32      0.32        22
                cotton_aphid       0.90      0.87      0.88        30
 cotton_calcium_deficiency_x       0.38      0.27      0.32        11
        cotton_fall_armyworm       0.90      0.90      0.90        48
  cotton_foot_and_collar_rot       0.74      0.56      0.64        41
         cotton_frost_damage       0.58      0.60      0.59        42
cotton_fusarium_wilt_disease       0.56      0.67      0.61        33
   cotton_leaf_miner_disease       0.69      0.63      0.66       

In [ ]:
# ✅ 7️⃣ Save Trained Model
torch.save(model.state_dict(), 'cotton_disease_classifier.pth')


In [ ]:
import os
from PIL import Image, UnidentifiedImageError
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

# =========================
# 1️⃣ Data Paths
# =========================
data_dir = '/content/drive/MyDrive/cotton'  # Update this path

# =========================
# 2️⃣ Remove Corrupt Images
# =========================
def remove_corrupt_images(data_dir):
    corrupt_files = []
    for root, _, files in os.walk(data_dir):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                img = Image.open(file_path)
                img.verify()
            except (IOError, SyntaxError, UnidentifiedImageError):
                print(f'Corrupt image found: {file_path}')
                corrupt_files.append(file_path)

    for file_path in corrupt_files:
        os.remove(file_path)
        print(f'Removed {file_path}')

remove_corrupt_images(data_dir)

# =========================
# 3️⃣ Data Transform & Loading (Augmentation + Normalization)
# =========================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

target_count = 500  # desired number of images per class


dataset = datasets.ImageFolder(root=data_dir, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# =========================
# 4️⃣ Model Definition
# =========================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = resnet18(weights=ResNet18_Weights.DEFAULT)
num_classes = len(dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# =========================
# 5️⃣ Loss & Optimizer
# =========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# =========================
# 6️⃣ Training Loop
# =========================
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Val Acc: {val_acc:.4f}')

# =========================
# 7️⃣ Save Trained Model
# =========================
torch.save(model.state_dict(), 'cotton_disease_classifier.pth')

# =========================
# 8️⃣ Inference Function
# =========================
def predict(image_path):
    image = Image.open(image_path)
    image = transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return dataset.classes[pred.item()]

# Example usage:
# disease = predict('/path/to/sample_leaf.jpg')
# print(f'Predicted disease: {disease}')


Epoch [1/20], Loss: 1.8667, Val Acc: 0.4922
Epoch [2/20], Loss: 1.1874, Val Acc: 0.5266
Epoch [3/20], Loss: 0.8889, Val Acc: 0.5455
Epoch [4/20], Loss: 0.7253, Val Acc: 0.5768
Epoch [5/20], Loss: 0.5735, Val Acc: 0.5862
Epoch [6/20], Loss: 0.4616, Val Acc: 0.5831
Epoch [7/20], Loss: 0.3764, Val Acc: 0.6082
Epoch [8/20], Loss: 0.3139, Val Acc: 0.5705
Epoch [9/20], Loss: 0.2781, Val Acc: 0.6176
Epoch [10/20], Loss: 0.2490, Val Acc: 0.5768
Epoch [11/20], Loss: 0.2376, Val Acc: 0.5925
Epoch [12/20], Loss: 0.2012, Val Acc: 0.6019
Epoch [13/20], Loss: 0.1956, Val Acc: 0.5893
Epoch [14/20], Loss: 0.1539, Val Acc: 0.6082
Epoch [15/20], Loss: 0.1492, Val Acc: 0.6082
Epoch [16/20], Loss: 0.1426, Val Acc: 0.5831
Epoch [17/20], Loss: 0.1331, Val Acc: 0.5831
Epoch [18/20], Loss: 0.1376, Val Acc: 0.6176
Epoch [19/20], Loss: 0.1348, Val Acc: 0.5956
Epoch [20/20], Loss: 0.1154, Val Acc: 0.5893


In [ ]:
import os
from PIL import Image, UnidentifiedImageError
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

# =========================
# 1️⃣ Data Paths
# =========================
data_dir = '/content/drive/MyDrive/soyabean'  # Update this path

# =========================
# 2️⃣ Remove Corrupt Images
# =========================
def remove_corrupt_images(data_dir):
    corrupt_files = []
    for root, _, files in os.walk(data_dir):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                img = Image.open(file_path)
                img.verify()
            except (IOError, SyntaxError, UnidentifiedImageError):
                print(f'Corrupt image found: {file_path}')
                corrupt_files.append(file_path)

    for file_path in corrupt_files:
        os.remove(file_path)
        print(f'Removed {file_path}')

remove_corrupt_images(data_dir)

# =========================
# 3️⃣ Data Augmentation & Loading
# =========================
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Load dataset
full_dataset = datasets.ImageFolder(root=data_dir, transform=transform_train)

# Train/Validation split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

# Replace validation transform
val_dataset.dataset.transform = transform_val

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# =========================
# 4️⃣ Model Definition
# =========================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = resnet18(weights=ResNet18_Weights.DEFAULT)
num_classes = len(full_dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# =========================
# 5️⃣ Loss & Optimizer
# =========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# =========================
# 6️⃣ Training Loop
# =========================
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Val Acc: {val_acc:.4f}')

# =========================
# 7️⃣ Save Trained Model
# =========================
torch.save(model.state_dict(), 'cucumber_disease_classifier.pth')

# =========================
# 8️⃣ Inference Function
# =========================
def predict(image_path):
    image = Image.open(image_path).convert('RGB')
    image = transform_val(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return full_dataset.classes[pred.item()]

# Example usage:
# disease = predict('/path/to/sample_leaf.jpg')
# print(f'Predicted disease: {disease}')


Corrupt image found: /content/drive/MyDrive/soyabean/Bacteria Pustle/Thumbs.db
Corrupt image found: /content/drive/MyDrive/soyabean/Frogeye Leaf Spot/Thumbs.db
Removed /content/drive/MyDrive/soyabean/Bacteria Pustle/Thumbs.db
Removed /content/drive/MyDrive/soyabean/Frogeye Leaf Spot/Thumbs.db
Epoch [1/10], Loss: 0.5921, Val Acc: 0.9252
Epoch [2/10], Loss: 0.0974, Val Acc: 0.9307
Epoch [3/10], Loss: 0.0468, Val Acc: 0.9612
Epoch [4/10], Loss: 0.0186, Val Acc: 0.9640
Epoch [5/10], Loss: 0.0175, Val Acc: 0.9612
Epoch [6/10], Loss: 0.0175, Val Acc: 0.9751
Epoch [7/10], Loss: 0.0148, Val Acc: 0.9806
Epoch [8/10], Loss: 0.0136, Val Acc: 0.9723
Epoch [9/10], Loss: 0.0052, Val Acc: 0.9806
Epoch [10/10], Loss: 0.0035, Val Acc: 0.9723
